In [1]:
import random
import numpy as np
import torch
from datasets import load_dataset
from sentence_transformers import SentenceTransformer, util
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix

seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)

device = "mps" if torch.backends.mps.is_available() else "cpu"
print(f"Using device: {device}")

Using device: mps


In [2]:
model_name = "sentence-transformers/paraphrase-MiniLM-L6-v2"
model = SentenceTransformer(model_name, device=device)
threshold = 0.5
print(f"Loaded model: {model_name}")
print(f"Fixed cosine-similarity threshold: {threshold}")

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Loaded model: sentence-transformers/paraphrase-MiniLM-L6-v2
Fixed cosine-similarity threshold: 0.5


In [3]:
dataset = load_dataset("glue", "mrpc", split="validation")
print("Dataset split: glue/mrpc validation")
print(f"Original number of examples: {len(dataset)}")
print("Example row:")
print(dataset[0])

label0_idxs = [i for i, x in enumerate(dataset) if x["label"] == 0]
label1_idxs = [i for i, x in enumerate(dataset) if x["label"] == 1]
balanced_count_per_label = min(len(label0_idxs), len(label1_idxs))

rng = random.Random(seed)
selected0 = rng.sample(label0_idxs, balanced_count_per_label)
selected1 = rng.sample(label1_idxs, balanced_count_per_label)
selected_indices = selected0 + selected1
rng.shuffle(selected_indices)

subset = dataset.select(selected_indices)
labels = subset["label"]
label_counts = {0: sum(1 for y in labels if y == 0), 1: sum(1 for y in labels if y == 1)}

print(f"Balanced subset size: {len(subset)}")
print(f"Balanced count per label: {balanced_count_per_label}")
print(f"Subset label counts: {label_counts}")

Dataset split: glue/mrpc validation
Original number of examples: 408
Example row:
{'sentence1': "He said the foodservice pie business doesn 't fit the company 's long-term growth strategy .", 'sentence2': '" The foodservice pie business does not fit our long-term growth strategy .', 'label': 1, 'idx': 9}
Balanced subset size: 258
Balanced count per label: 129
Subset label counts: {0: 129, 1: 129}


In [4]:
batch_size = 64
sentences1 = subset["sentence1"]
sentences2 = subset["sentence2"]

all_scores = []
all_predictions = []

for start_idx in range(0, len(subset), batch_size):
    end_idx = min(start_idx + batch_size, len(subset))
    batch_s1 = sentences1[start_idx:end_idx]
    batch_s2 = sentences2[start_idx:end_idx]
    emb1 = model.encode(batch_s1, batch_size=batch_size, convert_to_tensor=True, show_progress_bar=False)
    emb2 = model.encode(batch_s2, batch_size=batch_size, convert_to_tensor=True, show_progress_bar=False)
    scores = util.cos_sim(emb1, emb2).diagonal()
    preds = (scores >= threshold).long()
    all_scores.extend(scores.detach().cpu().tolist())
    all_predictions.extend(preds.detach().cpu().tolist())

print(f"Completed inference for {len(all_predictions)} examples.")
print(f"Similarity score range: min={min(all_scores):.4f}, max={max(all_scores):.4f}")

Completed inference for 258 examples.
Similarity score range: min=0.2963, max=0.9902


In [5]:
accuracy = accuracy_score(labels, all_predictions)
precision, recall, f1, _ = precision_recall_fscore_support(labels, all_predictions, average="binary", zero_division=0)
cm = confusion_matrix(labels, all_predictions)

tn, fp, fn, tp = cm.ravel()
per_label_error_counts = {
    0: int(fp),
    1: int(fn)
}

print("Subset evaluation metrics:")
print(f"Accuracy : {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1       : {f1:.4f}")
print("Confusion matrix:")
print(cm)
print("Per-label error counts:")
print({"label_0_errors": per_label_error_counts[0], "label_1_errors": per_label_error_counts[1]})

Subset evaluation metrics:
Accuracy : 0.5620
Precision: 0.5331
Recall   : 1.0000
F1       : 0.6954
Confusion matrix:
[[ 16 113]
 [  0 129]]
Per-label error counts:
{'label_0_errors': 113, 'label_1_errors': 0}


In [6]:
label_map = {0: "not_paraphrase", 1: "paraphrase"}
rows = []

for i in range(len(subset)):
    rows.append({
        "subset_idx": i,
        "score": float(all_scores[i]),
        "distance_to_threshold": abs(float(all_scores[i]) - threshold),
        "true_label": int(labels[i]),
        "pred_label": int(all_predictions[i]),
        "sentence1": subset[i]["sentence1"],
        "sentence2": subset[i]["sentence2"]
    })

ambiguous_pairs = sorted(rows, key=lambda x: (x["distance_to_threshold"], x["subset_idx"]))
num_examples_to_show = min(10, len(ambiguous_pairs))

print(f"Nearest-threshold ambiguous pairs: showing {num_examples_to_show} of {len(ambiguous_pairs)}")
for item in ambiguous_pairs[:num_examples_to_show]:
    print(f"Subset index: {item['subset_idx']}")
    print(f"Cosine similarity: {item['score']:.4f}")
    print(f"Distance to threshold: {item['distance_to_threshold']:.4f}")
    print(f"sentence1: {item['sentence1']}")
    print(f"sentence2: {item['sentence2']}")
    print(f"true label: {item['true_label']} ({label_map[item['true_label']]})")
    print(f"pred label: {item['pred_label']} ({label_map[item['pred_label']]})")
    print("-" * 80)

Nearest-threshold ambiguous pairs: showing 10 of 258
Subset index: 224
Cosine similarity: 0.5000
Distance to threshold: 0.0000
sentence1: However , EPA officials would not confirm the 20 percent figure .
sentence2: Only in the past few weeks have officials settled on the 20 percent figure .
true label: 0 (not_paraphrase)
pred label: 0 (not_paraphrase)
--------------------------------------------------------------------------------
Subset index: 129
Cosine similarity: 0.5012
Distance to threshold: 0.0012
sentence1: Lu reclined in a soft chair wearing a woolly coat near the blackened capsule .
sentence2: " It 's great to be back home , " said Lu , dressed in a woolly coat near the blackened capsule .
true label: 0 (not_paraphrase)
pred label: 1 (paraphrase)
--------------------------------------------------------------------------------
Subset index: 99
Cosine similarity: 0.5042
Distance to threshold: 0.0042
sentence1: Claire had advanced to the third round of the 76th annual Scripps How

In [7]:
print("RESULT SUMMARY")
print(f"model={model_name}")
print("dataset_split=glue/mrpc validation")
print("dataset_variant=balanced_subset_equal_labels")
print(f"device={device}")
print(f"threshold={threshold}")
print(f"num_examples={len(subset)}")
print(f"label_0_count={label_counts[0]}")
print(f"label_1_count={label_counts[1]}")
print(f"accuracy={accuracy:.4f}")
print(f"precision={precision:.4f}")
print(f"recall={recall:.4f}")
print(f"f1={f1:.4f}")
print(f"confusion_matrix={cm.tolist()}")
print(f"label_0_errors={per_label_error_counts[0]}")
print(f"label_1_errors={per_label_error_counts[1]}")

RESULT SUMMARY
model=sentence-transformers/paraphrase-MiniLM-L6-v2
dataset_split=glue/mrpc validation
dataset_variant=balanced_subset_equal_labels
device=mps
threshold=0.5
num_examples=258
label_0_count=129
label_1_count=129
accuracy=0.5620
precision=0.5331
recall=1.0000
f1=0.6954
confusion_matrix=[[16, 113], [0, 129]]
label_0_errors=113
label_1_errors=0
